# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading and exploration of the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the Croissant schema.

### Dataset Source
The dataset source is described and linked by a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure the latest mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and dataset objects from the FAIR² dataset using mlcroissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset: {metadata['name']}\n")
print(f"Description: {metadata['description']}\n")
print(f"Identifier: {metadata.get('identifier','')}\n")
print(f"Published: {metadata.get('datePublished','')}\n")
print(f"License: {metadata.get('license','')}\n")
print("Keywords:", metadata.get('keywords', []))

## 2. Data Overview
Review available record sets and their fields using their `@id`.

Croissant record sets may be large and have nested or grouped fields. We'll list all available record sets, along with sample fields and their types, identified by their `@id`.

In [ ]:
# List all record sets by their @id
print("Available record sets (@id):\n-----------------------------")
record_sets = list(dataset.record_sets)
for rset in record_sets:
    print(f"- @id: {rset.id}   -- name: {rset.name if hasattr(rset, 'name') else ''}")

# For each record set, print sample fields and their @id and data type
for rset in record_sets:
    print(f"\nFields for RecordSet @id: {rset.id}")
    for fld in rset.fields[:5]:  # Show first 5 fields only for brevity
        dtype = getattr(fld, 'data_type', None)
        field_name = getattr(fld, 'name', '')
        print(f"   - @id: {fld.id} | name: {field_name} | data_type: {dtype}")

## 3. Data Extraction
We'll extract tabular data from each available record set by referencing them via their unique `@id`. Each DataFrame loaded will be indexed by its record set `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rset.id for rset in dataset.record_sets]

print(f"\nExtracting data for record set(s):\n{', '.join(record_set_ids) if record_set_ids else 'None found in Croissant metadata.'}")

# For demonstration, extract records for all sets and print their columns
for rid in record_set_ids:
    recs = list(dataset.records(record_set=rid))
    df = pd.DataFrame(recs)
    dataframes[rid] = df
    print(f"\nRecord set @id: {rid}")
    print(f"  Columns: {list(df.columns)}")
    if len(df):
        display(df.head(3))
    else:
        print("  [No records loaded]")

## 4. Exploratory Data Analysis (EDA)
Now, we'll select a numeric field (by its `@id`) from the first available record set for demonstration. We show filtering, normalization, and grouping—all using the correct entity `@id` references.

In [ ]:
# For demonstration, use the first record set and first numeric field
# (Edit these IDs to fit actual numeric fields in your data)
import numpy as np

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Identify numeric fields by Croissant schema
    first_rset = next(filter(lambda r: r.id == record_set_id, dataset.record_sets), None)
    numeric_field_ids = [f.id for f in getattr(first_rset, 'fields', []) if getattr(f, 'data_type', None) in ('schema:Integer', 'schema:Float', 'schema:Number')]

    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
        print(f"Numeric field chosen: {numeric_field_id}")

        # Handle missing values if present and filter
        df = df.replace({numeric_field_id: {None: np.nan, '': np.nan}})
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        if filtered_df[numeric_field_id].notnull().any():
            normalized = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            filtered_df[f"{numeric_field_id}_normalized"] = normalized
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping if a categorical field exists
        group_field_ids = [f.id for f in getattr(first_rset, 'fields', []) if getattr(f, 'data_type', None) in ('schema:Text','schema:String')]
        if group_field_ids:
            group_field_id = group_field_ids[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f'{numeric_field_id}_mean')
                print(f"Grouped by {group_field_id}:")
                display(grouped_df.head())
    else:
        print("No numeric fields found in this record set. Please check the schema for numeric field @ids.")
else:
    print("No record sets available to analyze.")

## 5. Visualization
We will visualize the distribution of the chosen numeric field using matplotlib, referring to the field by its `@id`. More advanced visualizations (e.g., pairplots, boxplots) can be added as desired using these ids.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if data exists
if 'filtered_df' in locals() and numeric_field_id in filtered_df.columns and filtered_df[numeric_field_id].notnull().any():
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field data to plot.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using its Croissant schema, inspected all available record sets and fields by their `@id`, extracted data, and applied simple EDA and visualization. All references to entities used their Croissant `@id` identifiers, ensuring alignment with schema documentation and provenance.

**Next steps:** deeper analyses, joining additional record sets, and tailoring processing based on scientific questions—always using `@id` as the key linking attribute.